# Resolve 'Needs Human Review' Annotations

After majority voting (`majority_voting.ipynb`), any 3-way tie between the model annotations is flagged as `Needs Human Review`.

This notebook automatically resolves those flagged cells using keyword heuristics over the comment text (Nepali + English, migration-domain keywords), and saves the fully labeled '_Cleared' files. Any cell that cannot be resolved by the heuristics falls back to the dataset-dominant label for that column.

In [ ]:
import pandas as pd
from pathlib import Path

### Configure Data Paths
By default, paths are resolved relative to the repository root (this notebook lives in `src/annotation/`). The input is the majority-voted output, and the final files are saved with a `_Cleared` suffix, preserving the channel sub-folder structure.

In [ ]:
# Majority-voted annotations (may contain 'Needs Human Review' cells)
INPUT_DIR = Path('../..') / 'data' / 'annotations' / 'Majority_Voting'

# Directory where the final, fully labeled annotations will be saved
OUTPUT_DIR = Path('../..') / 'data' / 'annotations' / 'Final_Annotations'

# The annotation columns that may contain 'Needs Human Review'
LABEL_COLS = ['intent', 'primary_driver', 'value_orientation', 'affect']

print('--- Setup ---')
status = 'found' if INPUT_DIR.exists() else 'NOT FOUND - check the path!'
print(f'Input directory: {INPUT_DIR} ({status})')
print(f'Output directory: {OUTPUT_DIR}')

### Heuristic Resolver
For each flagged cell, the comment text is matched against keyword lists:

- **intent**: leaving/returning phrases decide Pro- vs Anti-Migration, otherwise Neutral/Observation.
- **primary_driver**: politics/government keywords → Systemic/Political Anger; family → Family Obligation; jobs/money → Economic Necessity; love of country → Patriotism/Love; fallback is Economic Necessity (the dominant driver in this dataset).
- **value_orientation**: family → Collectivist-Family; nation → Collectivist-Nation; fallback Individualist-Self.
- **affect**: sadness, hope, or anger keywords; fallback Pragmatic.

Comments that are *meta* (about the video/podcast itself), about language choice, or factual corrections are overridden to Neutral/Observation since they do not express a real migration stance.

In [ ]:
def auto_fix_review(row):
    comm = str(row['comment']).lower()
    intent = str(row['intent'])
    driver = str(row['primary_driver'])
    orient = str(row['value_orientation'])
    affect = str(row['affect'])

    # Context flags
    is_meta = any(k in comm for k in ['like your video', 'nice video', 'good podcast', 'great podcast', 'podcast is good', 'educational and entertaining', 'types of communication', 'repost', 'very nice', 'face reveal', 'best content', 'quality content'])
    is_lang = any(k in comm for k in ['speak full english', 'purai english', 'nepali mai bolnu', 'language', 'bhaso', 'bhasa'])
    is_fact_corr = any(k in comm for k in ['3 million', '30 million', '3million', '30million', '3 crore', '3crore', 'wrong data', 'data mistake', 'data is wrong'])
    is_neta_sarkar = any(k in comm for k in ['neta', 'sarkar', 'corruption', 'bhrastachar', 'pm', 'oli', 'deuba', 'prachanda', 'politician', 'government'])
    is_family = any(k in comm for k in ['family', 'baba', 'mami', 'pariwar', 'aama', 'buwa', 'parents', 'xora', 'xori'])
    is_job_money = any(k in comm for k in ['job', 'unemployment', 'salary', 'paisa', 'money', 'earn', 'income', 'tu ', 'university', 'bachelor', 'it sector', 'freelance'])

    # Fix Intent
    if intent == 'Needs Human Review':
        if any(k in comm for k in ['ma pani janchu', 'leaving', 'process gari', 'flight', 'apply gare', 'bidesh nai last option', 'fly away']):
            intent = 'Pro-Migration'
        elif any(k in comm for k in ['nepal mai bas', 'deshmai', 'yahi kei gar', 'farkera', 'return', 'chodne xaina', 'bidesh jana man xaina']):
            intent = 'Anti-Migration'
        else:
            intent = 'Neutral/Observation'

    # Fix Primary Driver
    if driver == 'Needs Human Review':
        if is_neta_sarkar:
            driver = 'Systemic/Political Anger'
        elif is_family:
            driver = 'Family Obligation'
        elif is_job_money:
            driver = 'Economic Necessity'
        elif 'maya' in comm or 'love' in comm or 'desh ko' in comm:
            driver = 'Patriotism/Love'
        else:
            driver = 'Economic Necessity'  # Baseline dominant driver in dataset

    # Fix Value Orientation
    if orient == 'Needs Human Review':
        if is_family:
            orient = 'Collectivist-Family'
        elif 'desh' in comm or 'nation' in comm or 'nepali' in comm:
            orient = 'Collectivist-Nation'
        else:
            orient = 'Individualist-Self'

    # Fix Affect
    if affect == 'Needs Human Review':
        if any(k in comm for k in ['aasu', 'crying', 'sad', 'regret', 'depressed', 'pida', 'hell', 'struggle', 'lost']):
            affect = 'Despairing/Sad'
        elif any(k in comm for k in ['hope', 'motivate', 'optimistic', 'will change', 'bright', 'rock', 'proud']):
            affect = 'Hopeful/Motivated'
        elif any(k in comm for k in ['wtf', 'fuck', 'shitty', 'chor', 'daka', 'khatey', 'muji', 'nonsense', 'suck', 'angry']):
            affect = 'Angry/Frustrated'
        else:
            affect = 'Pragmatic'

    # Post-processing override for meta context rows that don't apply to real migration drivers
    if is_meta or is_lang or is_fact_corr:
        if intent in ('Needs Human Review', 'Pro-Migration', 'Anti-Migration'):
            if not any(k in comm for k in ['ma janchu', 'desh xadney']):
                intent = 'Neutral/Observation'
        if driver in ['Economic Necessity', 'Systemic/Political Anger', 'Patriotism/Love', 'Family Obligation']:
            if not (is_neta_sarkar or is_job_money or is_family):
                driver = 'Economic Necessity'  # Default general classification or keep standard baseline

    return pd.Series([intent, driver, orient, affect])

### Execution
Loops through every majority-voted CSV, resolves the flagged cells, verifies that no `Needs Human Review` tags remain, and saves the cleared file (renaming `_Annotated` → `_Cleared`).

In [ ]:
if not INPUT_DIR.exists():
    print(f"Error: Input directory '{INPUT_DIR}' does not exist. Run majority_voting.ipynb first.")
else:
    files_processed = 0
    total_resolved = 0

    for file_path in sorted(INPUT_DIR.rglob('*.csv')):
        rel_path = file_path.relative_to(INPUT_DIR)
        df = pd.read_csv(file_path)

        missing_cols = [c for c in LABEL_COLS if c not in df.columns]
        if missing_cols:
            print(f'Skipping {rel_path}: missing columns {missing_cols}.')
            continue

        # Count flagged cells before resolving
        flagged = int((df[LABEL_COLS] == 'Needs Human Review').sum().sum())

        # Apply the heuristic resolver row by row
        df[LABEL_COLS] = df.apply(auto_fix_review, axis=1)

        # Verify nothing was left unresolved
        remaining = int((df[LABEL_COLS] == 'Needs Human Review').sum().sum())
        if remaining > 0:
            print(f'Warning: {remaining} cells still flagged in {rel_path}!')

        # Save with the '_Cleared' suffix, keeping the sub-folder structure
        if '_Annotated' in rel_path.name:
            out_name = rel_path.name.replace('_Annotated', '_Cleared')
        else:
            out_name = rel_path.stem + '_Cleared.csv'
        out_file = OUTPUT_DIR / rel_path.parent / out_name
        out_file.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_file, index=False)

        files_processed += 1
        total_resolved += flagged - remaining
        print(f'Processed and saved: {out_file} ({flagged} flagged cells resolved)')

    print(f"\n\u2705 Review resolution complete! {files_processed} files saved, "
          f"{total_resolved} 'Needs Human Review' cells resolved.")